## 下载模型

## 数据处理

In [1]:
import json5

data = []
with open('relationship_output_type.jsonl', 'r') as f:
    data = [eval(i.strip()) for i in f.readlines()]

In [2]:
data[0]

{'instruction': '你是一个中文文本关系抽取领域的专家，请从给定的文本中抽取出三元组（subject, predicate, object），并输出为 JSON 格式。\n本任务支持的 predicate 类型及其适用的实体类型组合如下：\n1. 体验内容：\n   - subject 实体类型 ∈ [目的地, 餐饮, 住宿]\n   - object 实体类型 ∈ [产品]\n2. 预算限制：\n   - subject 实体类型 ∈ [目的地, 餐饮, 住宿, 产品, 交通]\n   - object 实体类型 ∈ [预算]\n3. 人均预算：\n   - subject 实体类型 ∈ [预算]\n   - object 实体类型 ∈ [人数]\n4. 停留时长：\n   - subject 实体类型 ∈ [目的地, 餐饮, 住宿, 产品]\n   - object 实体类型 ∈ [时长]\n5. 出行时间：\n   - subject 实体类型 ∈ [出发地, 返程地, 目的地, 餐饮, 住宿, 产品]\n   - object 实体类型 ∈ [时间]\n6. 游玩顺序（路径类关系）：\n   - ① subject 实体类型 ∈ [出发地]，object 实体类型 ∈ [目的地]\n   - ② subject 实体类型 ∈ [目的地]，object 实体类型 ∈ [目的地]\n   - ③ subject 实体类型 ∈ [目的地]，object 实体类型 ∈ [返程地]\n请严格按照以下要求执行：\n- 所有三元组必须遵循上述实体类型对；\n- 如果一个句子中有多个符合的三元组，请全部提取；\n- 输出格式如下：{"subject": "青海", "predicate": "到达时间", "object": "10月22日"},{"subject": "日本","predicate": "停留时长","object": "8天"}',
 'input': '文本:求光雾山2日游攻略，主要是住宿和路线～重庆过去，打算周五下午出发，第一晚是直接到光雾山镇住比较好，还是住南江县城（ps看了下光雾山镇住宿就餐环境较差）？第二晚准备住光雾山镇，有好的价格在300元左右的酒店推荐吗？第三天也就是周日下午返回最后求个各个景区的游玩路线',
 'output

In [3]:
import json

In [4]:
new_data = [
    {'text':d['input'], 'ralations':[
        [di['subject'], 
         di['predicate'], 
         di['object']] for di in d['output']]
    }
    for d in data
]
new_data[0]

{'text': '文本:求光雾山2日游攻略，主要是住宿和路线～重庆过去，打算周五下午出发，第一晚是直接到光雾山镇住比较好，还是住南江县城（ps看了下光雾山镇住宿就餐环境较差）？第二晚准备住光雾山镇，有好的价格在300元左右的酒店推荐吗？第三天也就是周日下午返回最后求个各个景区的游玩路线',
 'ralations': [['南江县城', '预算限制', '300元'],
  ['光雾山镇', '预算限制', '300元'],
  ['光雾山镇', '出行时间', '第二晚'],
  ['光雾山', '出行时间', '周五下午'],
  ['光雾山', '停留时长', '2天'],
  ['光雾山镇', '出行时间', '第一晚'],
  ['南江县城', '出行时间', '第一晚'],
  ['光雾山', '出行时间', '周日下午']]}

In [5]:
prompt = """
你是一个中文文本关系抽取领域的专家，请仔细分析以下文本，严格按以下要求执行：

要求：
一.识别所有的关系数据,有如下关系类型：游玩顺序,出行时间,停留时长,预算限制,人均预算,体验内容。
二.关系对应的实体类型组合如下：
    1. 体验内容：
    - ob1 entity_label ∈ [目的地, 餐饮, 住宿]
    - ob2 entity_label ∈ [产品]
    2. 预算限制：
   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品, 交通]
   - ob2 entity_label ∈ [预算]
    3. 人均预算：
   - ob1 entity_label ∈ [预算]
   - ob2 entity_label ∈ [人数]
    4. 停留时长：
   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品]
   - ob2 entity_label ∈ [时长]
    5. 出行时间：
   - ob1 entity_label ∈ [出发地, 返程地, 目的地, 餐饮, 住宿, 产品]
   - ob2 entity_label ∈ [时间]
    6. 游玩顺序：
   - ① ob1 entity_label ∈ [出发地]，ob2 实体类型 ∈ [目的地]
   - ② ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [目的地]
   - ③ ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [返程地]
三.所有三元组必须遵循上述实体类型对
四.输出格式为JSON列表，每个元素包含三个字段："ob1", "rel", "ob2"

输出示例：
[
  {"ob1": "x", "rel": "x", 'ob2':'x'},
  {"ob1": "x", "rel": "x", 'ob2':'x'},
]

若无关系则输出:"找不到关系"

请处理以下文本：

{text}

""".strip()

def _format(d):
    d = json.dumps(
        [
            {'ob1':di[0], 'rel':di[1], 'ob2':di[2]} 
            for di in d['ralations']
        ], 
        ensure_ascii=False, 
        indent=2
    )
    d = f'```json\n{d}\n```'
    return d

messages_list = [
    {
        'messages':[
            {'role':'user', 'content':prompt.replace('{text}', d['text'])},
            {'role':'assistant', 'content': _format(d)},
        ]
    }
    for d in new_data
]

In [6]:
len(messages_list), messages_list[0]

(399,
 {'messages': [{'role': 'user',
    'content': '你是一个中文文本关系抽取领域的专家，请仔细分析以下文本，严格按以下要求执行：\n\n要求：\n一.识别所有的关系数据,有如下关系类型：游玩顺序,出行时间,停留时长,预算限制,人均预算,体验内容。\n二.关系对应的实体类型组合如下：\n    1. 体验内容：\n    - ob1 entity_label ∈ [目的地, 餐饮, 住宿]\n    - ob2 entity_label ∈ [产品]\n    2. 预算限制：\n   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品, 交通]\n   - ob2 entity_label ∈ [预算]\n    3. 人均预算：\n   - ob1 entity_label ∈ [预算]\n   - ob2 entity_label ∈ [人数]\n    4. 停留时长：\n   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品]\n   - ob2 entity_label ∈ [时长]\n    5. 出行时间：\n   - ob1 entity_label ∈ [出发地, 返程地, 目的地, 餐饮, 住宿, 产品]\n   - ob2 entity_label ∈ [时间]\n    6. 游玩顺序：\n   - ① ob1 entity_label ∈ [出发地]，ob2 实体类型 ∈ [目的地]\n   - ② ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [目的地]\n   - ③ ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [返程地]\n三.所有三元组必须遵循上述实体类型对\n四.输出格式为JSON列表，每个元素包含三个字段："ob1", "rel", "ob2"\n\n输出示例：\n[\n  {"ob1": "x", "rel": "x", \'ob2\':\'x\'},\n  {"ob1": "x", "rel": "x", \'ob2\':\'x\'},\n]\n\n若无关系则输出:"找不到关系"\n\n请处理以下文本：\n\n文本:求光雾山2日游攻略，主要是住宿和路线～重庆过去，打算

In [7]:
import random

random.shuffle(messages_list)
random.shuffle(messages_list)
random.shuffle(messages_list)

train_nums = int(len(messages_list) * 0.8)

In [8]:
with open('train_t.json', 'w', encoding='utf-8') as f:
    json.dump(messages_list[:train_nums], f, ensure_ascii=False, indent=2)

In [9]:
with open('valid_t.json', 'w', encoding='utf-8') as f:
    json.dump(messages_list[train_nums:], f, ensure_ascii=False, indent=2)

In [10]:
with open('results_t.json', 'w', encoding='utf-8') as f:
    json.dump(messages_list[train_nums:], f, ensure_ascii=False, indent=2)

## 微调

In [13]:
!swift sft \
--model models/Qwen2.5-7B-Instruct \
--model_type qwen2_5 \
--dataset train_t.json \
--val_dataset valid_t.json \
--logging_steps 5 \
--save-steps 10 \
--learning_rate 3e-5 \
--output_dir output \
--lora_rank 8 \
--lora_alpha 16 \
--lora_dropout 0. \
--weight_decay 0.01 \
--num_train_epochs 2 \
--per_device_train_batch_size 2 \
--gradient_accumulation_steps 2 \
--warmup-ratio 0.01 \
--eval_steps 20 \
--evaluation-strategy steps \
--max-grad-norm 1.0 \
--lr-scheduler-type cosine \
--save-total-limit 3 \
--max_length 2000 \
--truncation_strategy delete

run sh: `/root/miniconda3/bin/python /root/miniconda3/lib/python3.12/site-packages/swift/cli/sft.py --model models/Qwen2.5-7B-Instruct --model_type qwen2_5 --dataset train_t.json --val_dataset valid_t.json --logging_steps 5 --save-steps 10 --learning_rate 3e-5 --output_dir output --lora_rank 8 --lora_alpha 16 --lora_dropout 0. --weight_decay 0.01 --num_train_epochs 2 --per_device_train_batch_size 2 --gradient_accumulation_steps 2 --warmup-ratio 0.01 --eval_steps 20 --evaluation-strategy steps --max-grad-norm 1.0 --lr-scheduler-type cosine --save-total-limit 3 --max_length 2000 --truncation_strategy delete`
[INFO:swift] Successfully registered `/root/miniconda3/lib/python3.12/site-packages/swift/llm/dataset/data/dataset_info.json`.
[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_size: 1
[INFO:swift] Loading the model using model_dir: models/Qwen2.5-7B-Instruct
[INFO:swift] Setting torch_dtype: torch.bfloat16
[INFO:swift] Because len(args.val_dataset) > 0, setting split

## 用vllm推理部署

xxx 替换成训练后保存的 output/xxx/xxx

!!import !! 终端运行

In [ ]:
vllm serve models/Qwen2.5-7B-Instruct \
--enable-lora \
--lora-modules qwen2.5-lora-t=output/v2-20250330-041354/checkpoint-158 \
--max-model-len 2000 \
--dtype bfloat16

INFO 03-29 23:53:49 [__init__.py:239] Automatically detected platform cuda.
INFO 03-29 23:53:51 [api_server.py:981] vLLM API server version 0.8.2
INFO 03-29 23:53:51 [api_server.py:982] args: Namespace(subparser='serve', model_tag='models/Qwen2.5-7B-Instruct', config='', host=None, port=8000, uvicorn_log_level='info', disable_uvicorn_access_log=False, allow_credentials=False, allowed_origins=['*'], allowed_methods=['*'], allowed_headers=['*'], api_key=None, lora_modules=[LoRAModulePath(name='qwen2.5-lora', path='output/v0-20250329-231243/checkpoint-140', base_model_name=None)], prompt_adapters=None, chat_template=None, chat_template_content_format='auto', response_role='assistant', ssl_keyfile=None, ssl_certfile=None, ssl_ca_certs=None, enable_ssl_refresh=False, ssl_cert_reqs=0, root_path=None, middleware=[], return_tokens_as_token_ids=False, disable_frontend_multiprocessing=False, enable_request_id_headers=False, enable_auto_tool_choice=False, tool_call_parser=None, tool_parser_plugin

## 跑验证集数据


In [14]:
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1",
)
client.models.list()

SyncPage[Model](data=[Model(id='models/Qwen2.5-7B-Instruct', created=1743280130, object='model', owned_by='vllm', root='models/Qwen2.5-7B-Instruct', parent=None, max_model_len=2000, permission=[{'id': 'modelperm-6e03b6dab5ca4976a37c115ec79805e0', 'object': 'model_permission', 'created': 1743280130, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]), Model(id='qwen2.5-lora-t', created=1743280130, object='model', owned_by='vllm', root='output/v2-20250330-041354/checkpoint-158', parent='models/Qwen2.5-7B-Instruct', max_model_len=None, permission=[{'id': 'modelperm-b77cb32780294e7b93efc09d65faa677', 'object': 'model_permission', 'created': 1743280130, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 

In [26]:
import json

with open('results_t.json', 'r') as f:
    valid_data = json.load(f)
len(valid_data), valid_data[0]

(80,
 {'messages': [{'role': 'user',
    'content': '你是一个中文文本关系抽取领域的专家，请仔细分析以下文本，严格按以下要求执行：\n\n要求：\n一.识别所有的关系数据,有如下关系类型：游玩顺序,出行时间,停留时长,预算限制,人均预算,体验内容。\n二.关系对应的实体类型组合如下：\n    1. 体验内容：\n    - ob1 entity_label ∈ [目的地, 餐饮, 住宿]\n    - ob2 entity_label ∈ [产品]\n    2. 预算限制：\n   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品, 交通]\n   - ob2 entity_label ∈ [预算]\n    3. 人均预算：\n   - ob1 entity_label ∈ [预算]\n   - ob2 entity_label ∈ [人数]\n    4. 停留时长：\n   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品]\n   - ob2 entity_label ∈ [时长]\n    5. 出行时间：\n   - ob1 entity_label ∈ [出发地, 返程地, 目的地, 餐饮, 住宿, 产品]\n   - ob2 entity_label ∈ [时间]\n    6. 游玩顺序：\n   - ① ob1 entity_label ∈ [出发地]，ob2 实体类型 ∈ [目的地]\n   - ② ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [目的地]\n   - ③ ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [返程地]\n三.所有三元组必须遵循上述实体类型对\n四.输出格式为JSON列表，每个元素包含三个字段："ob1", "rel", "ob2"\n\n输出示例：\n[\n  {"ob1": "x", "rel": "x", \'ob2\':\'x\'},\n  {"ob1": "x", "rel": "x", \'ob2\':\'x\'},\n]\n\n若无关系则输出:"找不到关系"\n\n请处理以下文本：\n\n文本:求推荐旅游计划书求推荐旅游计划书，从常州出发目的地深圳

In [29]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

prompt = """
你是一个中文文本关系抽取领域的专家，请仔细分析以下文本，严格按以下要求执行：

要求：
一.识别所有的关系数据,有如下关系类型：游玩顺序,出行时间,停留时长,预算限制,人均预算,体验内容。
二.关系对应的实体类型组合如下：
    1. 体验内容：
    - ob1 entity_label ∈ [目的地, 餐饮, 住宿]
    - ob2 entity_label ∈ [产品]
    2. 预算限制：
   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品, 交通]
   - ob2 entity_label ∈ [预算]
    3. 人均预算：
   - ob1 entity_label ∈ [预算]
   - ob2 entity_label ∈ [人数]
    4. 停留时长：
   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品]
   - ob2 entity_label ∈ [时长]
    5. 出行时间：
   - ob1 entity_label ∈ [出发地, 返程地, 目的地, 餐饮, 住宿, 产品]
   - ob2 entity_label ∈ [时间]
    6. 游玩顺序：
   - ① ob1 entity_label ∈ [出发地]，ob2 实体类型 ∈ [目的地]
   - ② ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [目的地]
   - ③ ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [返程地]
三.所有三元组必须遵循上述实体类型对
四.输出格式为JSON列表，每个元素包含三个字段："ob1", "rel", "ob2"

输出示例：
[
  {"ob1": "x", "rel": "x", 'ob2':'x'},
  {"ob1": "x", "rel": "x", 'ob2':'x'},
]

若无关系则输出:"找不到关系"

请处理以下文本：

{text}

""".strip()

def process(idx:int, msg: dict):
    api_key = "XXX"
    base_url = "http://localhost:8000/v1"
    model = "models/Qwen2.5-7B-Instruct"
    # model='qwen2.5-lora-t'
    # model = 'models/glm-4-9b-chat'
    # model = 'glm4-lora'
    # model = 'models/internlm3-8b-instruct'
    # model = 'internlm3-lora'
    temperature = 0.
    client = OpenAI(api_key=api_key, base_url=base_url)
    
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": prompt.replace("{text}", msg["messages"][0]['content'])},
            ],
            temperature=temperature,
        )
        content = resp.choices[0].message.content
        msg[model] = content
    except:
        msg[model] = ''
    
    return idx, msg

max_workers = 60
start_time = time.time()
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [
        executor.submit(process, idx, msg) for idx, msg in enumerate(valid_data)
    ]

    num = 0
    print(f'>all {len(futures)} <<<<')
    for future in as_completed(futures):
        idx, msg = future.result()
        valid_data[idx] = msg

        if (num :=num + 1) and (num % max_workers==0 or num == len(futures)):
            print(f">> {time.strftime('%m-%d %H:%M:%S')} >> {num:<5} << {time.time()-start_time:.3f}s")

            with open('results.json', 'w', encoding='utf-8') as f:
                json.dump(valid_data, f, ensure_ascii=False, indent=2)
            start_time = time.time()

with open('results_t.json', 'w', encoding='utf-8') as f:
    json.dump(valid_data, f, ensure_ascii=False, indent=2)

>all 80 <<<<
>> 03-30 04:34:35 >> 60    << 35.597s
>> 03-30 04:34:41 >> 80    << 6.178s


In [30]:
valid_data[0].keys()

dict_keys(['messages', 'qwen2.5-lora-t', 'models/Qwen2.5-7B-Instruct'])

## 评估

In [31]:
import json

with open('results_t.json', 'r') as f:
    results = json.load(f)
results[0]

{'messages': [{'role': 'user',
   'content': '你是一个中文文本关系抽取领域的专家，请仔细分析以下文本，严格按以下要求执行：\n\n要求：\n一.识别所有的关系数据,有如下关系类型：游玩顺序,出行时间,停留时长,预算限制,人均预算,体验内容。\n二.关系对应的实体类型组合如下：\n    1. 体验内容：\n    - ob1 entity_label ∈ [目的地, 餐饮, 住宿]\n    - ob2 entity_label ∈ [产品]\n    2. 预算限制：\n   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品, 交通]\n   - ob2 entity_label ∈ [预算]\n    3. 人均预算：\n   - ob1 entity_label ∈ [预算]\n   - ob2 entity_label ∈ [人数]\n    4. 停留时长：\n   - ob1 entity_label ∈ [目的地, 餐饮, 住宿, 产品]\n   - ob2 entity_label ∈ [时长]\n    5. 出行时间：\n   - ob1 entity_label ∈ [出发地, 返程地, 目的地, 餐饮, 住宿, 产品]\n   - ob2 entity_label ∈ [时间]\n    6. 游玩顺序：\n   - ① ob1 entity_label ∈ [出发地]，ob2 实体类型 ∈ [目的地]\n   - ② ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [目的地]\n   - ③ ob1 entity_label ∈ [目的地]，ob2 实体类型 ∈ [返程地]\n三.所有三元组必须遵循上述实体类型对\n四.输出格式为JSON列表，每个元素包含三个字段："ob1", "rel", "ob2"\n\n输出示例：\n[\n  {"ob1": "x", "rel": "x", \'ob2\':\'x\'},\n  {"ob1": "x", "rel": "x", \'ob2\':\'x\'},\n]\n\n若无关系则输出:"找不到关系"\n\n请处理以下文本：\n\n文本:求推荐旅游计划书求推荐旅游计划书，从常州出发目的地深圳，沿途看风景可

- 计算acc pre f1

In [32]:
def calculate_metrics(preds:list[str], targets:list[str]):
    preds_set, targets_set = set(preds), set(targets)
    
    tp = len(preds_set & targets_set)  # True Positives
    fp = len(preds_set - targets_set)  # False Positives
    fn = len(targets_set - preds_set)  # False Negatives
    
    acc = tp / max(len(targets_set), len(preds_set)) if max(len(targets_set), len(preds_set)) > 0 else 0
    pre = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * pre * rec / (pre + rec) if (pre + rec) > 0 else 0
    
    return acc, pre,rec, f1


In [33]:
models = [
    'models/Qwen2.5-7B-Instruct', 
    'qwen2.5-lora-t'
]
for model in models:
    scores = []
    enums = 0
    for d in results:
        t = d['messages'][1]['content']
        tt = [[i['ob1'], i['rel'],i['ob2']] for i in eval(t[7:-3])]
        targets = list(set(sum(tt, [])))

        try:
            jd = d[model].replace('null', '""').strip()
            if '```json' in jd:
                si = jd.index('```json')+7
            else:
                si = 0
            if '```' in jd[si:]:
                ei = jd.rindex('```')
            else:
                ei = len(jd)
            jd = jd[si:ei]
            
            tp = [[i['ob1'], i['rel'],i['ob2']] for i in eval(jd)]
            preds = list(set(sum(tp, [])))
            
            # 
            score = calculate_metrics(preds, targets)
            scores.append(score)
        except:
            enums +=1

    accs = [i[0] for i in scores]
    pres = [i[1] for i in scores]
    recs = [i[2] for i in scores]
    f1s  = [i[3] for i in scores]
    print('model:', model)
    print('enums:', enums)
    print('acc:', sum(accs)/len(accs))
    print('pre:', sum(pres)/len(pres))
    print('rec:', sum(recs)/len(recs))
    print('f1:', sum(f1s)/len(f1s))
    print('==')

model: models/Qwen2.5-7B-Instruct
enums: 15
acc: 0.37702039162446405
pre: 0.39073223533630774
rec: 0.5864346764346764
f1: 0.4479161628415556
==
model: qwen2.5-lora-t
enums: 0
acc: 0.6948015873015873
pre: 0.7742559523809524
rec: 0.7207539682539682
f1: 0.7320562770562771
==
